# DFU Data Prep (Light Augmentation)

This notebook creates a lightly-augmented training split from:
`/kaggle/input/datasets/yessinehakim1/dfu-seg-new/DFU Seg`

Output is written to:
`/kaggle/working/DFU Seg`

You can zip and upload it as a Kaggle dataset named `dfu-seg-preped`.


In [1]:
from pathlib import Path
import os
import cv2
import numpy as np
from tqdm import tqdm
import albumentations as A
import shutil


In [2]:
SRC_ROOT = Path('/kaggle/input/datasets/yessinehakim1/dfu-seg-new/DFU Seg')
DST_ROOT = Path('/kaggle/working/DFU Seg')

assert SRC_ROOT.exists(), f'Missing source dataset: {SRC_ROOT}'

if DST_ROOT.exists():
    shutil.rmtree(DST_ROOT)

for split in ['train', 'val', 'test']:
    (DST_ROOT / split / 'images').mkdir(parents=True, exist_ok=True)
    (DST_ROOT / split / 'masks').mkdir(parents=True, exist_ok=True)

print('SRC:', SRC_ROOT)
print('DST:', DST_ROOT)


SRC: /kaggle/input/datasets/yessinehakim1/dfu-seg-new/DFU Seg
DST: /kaggle/working/DFU Seg


In [3]:
light_aug = A.Compose([
    A.HorizontalFlip(p=0.6),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.6),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.08, rotate_limit=15,
                       interpolation=cv2.INTER_LINEAR,
                       border_mode=cv2.BORDER_REFLECT_101,
                       p=0.6),
])

def read_image(path):
    img = cv2.imread(str(path), cv2.IMREAD_COLOR)
    if img is None:
        raise ValueError(f'Cannot read image: {path}')
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def read_mask(path):
    m = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if m is None:
        raise ValueError(f'Cannot read mask: {path}')
    return m

def save_image(path, img_rgb):
    img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)
    cv2.imwrite(str(path), img_bgr)

def save_mask(path, m):
    cv2.imwrite(str(path), m)


/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [4]:
# How many augmented copies per validation/test image.
# Kept moderate to stay reasonably light.
AUG_PER_IMAGE = 2


In [5]:
def list_pairs(split):
    img_dir = SRC_ROOT / split / 'images'
    msk_dir = SRC_ROOT / split / 'masks'

    imgs = sorted([p for p in img_dir.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png'}])
    pairs = []
    for ip in imgs:
        mp = msk_dir / f'{ip.stem}.png'
        if mp.exists():
            pairs.append((ip, mp))
    return pairs

for split in ['train', 'val', 'test']:
    pairs = list_pairs(split)
    print(split, len(pairs))


train 707
val 152
test 151


In [6]:
for split in ['train', 'val', 'test']:
    pairs = list_pairs(split)
    out_img = DST_ROOT / split / 'images'
    out_msk = DST_ROOT / split / 'masks'

    for ip, mp in tqdm(pairs, desc=f'Copy/Augment {split}'):
        img = read_image(ip)
        msk = read_mask(mp)

        # Always keep original
        save_image(out_img / ip.name, img)
        save_mask(out_msk / mp.name, msk)

        # Apply offline augmentation for all splits
        for k in range(AUG_PER_IMAGE):
            aug = light_aug(image=img, mask=msk)
            aimg, amsk = aug['image'], aug['mask']

            aug_img_name = f'{ip.stem}__aug{k}.png'
            aug_msk_name = f'{mp.stem}__aug{k}.png'

            save_image(out_img / aug_img_name, aimg)
            save_mask(out_msk / aug_msk_name, amsk)

print('Done')


Copy/Augment test: 100%|██████████| 151/151 [00:11<00:00, 12.86it/s]

Done


In [7]:
# Optional: package for upload as a Kaggle dataset
!cd /kaggle/working && zip -r "dfu-seg-preped.zip" "DFU Seg"


updating: DFU Seg/ (stored 0%)
updating: DFU Seg/test/ (stored 0%)
updating: DFU Seg/test/images/ (stored 0%)
updating: DFU Seg/val/ (stored 0%)
updating: DFU Seg/val/images/ (stored 0%)
updating: DFU Seg/train/ (stored 0%)
updating: DFU Seg/train/images/ (stored 0%)
updating: DFU Seg/test/images/0861.png (deflated 6%)
updating: DFU Seg/test/images/0680.png (deflated 5%)
updating: DFU Seg/test/images/0170.png (deflated 7%)
updating: DFU Seg/test/images/0047.png (deflated 4%)
updating: DFU Seg/test/images/0097.png (deflated 6%)
updating: DFU Seg/test/images/0090.png (deflated 8%)
updating: DFU Seg/test/images/0692.png (deflated 6%)
updating: DFU Seg/test/images/1003.png (deflated 5%)
updating: DFU Seg/test/images/0068.png (deflated 8%)
updating: DFU Seg/test/images/0929.png (deflated 6%)
updating: DFU Seg/test/images/0579.png (deflated 8%)
updating: DFU Seg/test/images/0777.png (deflated 6%)
updating: DFU Seg/test/images/0221.png (deflated 7%)
updating: DFU Seg/test/images/0845.png (def